## Counting Bottles using Yolo

In [ ]:
import cv2
from ultralytics import YOLO

video_file_path = r"C:\Users\s86\Desktop\learning opencv\battles.mp4"
model_name = "yolov5su"  # Use the fine-tuned YOLO model

# Load your fine-tuned YOLO model
model = YOLO(model_name)  # Replace with your own model

# Open the video file
cap = cv2.VideoCapture(video_file_path)

if not cap.isOpened():
    print("Unable to open the video file!")
    exit()

# Define a vertical line for counting (example at x = 400)
threshold_line_x = 400

# Initialize tracking data
counted_bottles = set()  # This will hold xmin values of bottles that have crossed
bottle_count = 0
frame_count = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # Perform inference on the frame (detection)
    results = model(frame)

    # Access results for the first frame
    result = results[0]

    # Extract bounding boxes and class names
    boxes = result.boxes.xyxy  # Bounding boxes (x1, y1, x2, y2)
    confidences = result.boxes.conf  # Confidence score for each detection
    class_ids = result.boxes.cls  # Class IDs for the detections
    names = result.names  # Class names (like 'person', 'bottle', etc.)

    # Prepare detections for tracking (only bottles)
    for i in range(len(boxes)):
        class_id = class_ids[i].item()
        if names[class_id] == "bottle":  # Check if the detected class is 'bottle'
            x1, y1, x2, y2 = boxes[i].tolist()

            # Check if the bottle's xmin (leftmost side of the bounding box) is within the threshold range
            if threshold_line_x < x1 <= threshold_line_x + 4 and x1 not in counted_bottles:
                # Bottle crosses the threshold range and has not been counted before
                counted_bottles.add(x1)  # Use xmin as a unique identifier
                bottle_count += 1  # Increment bottle count

    # Draw the threshold line on the frame
    cv2.line(frame, (threshold_line_x, 0), (threshold_line_x, frame.shape[0]), (0, 255, 0), 2)

    # Draw bounding boxes and labels for detected bottles
    for i in range(len(boxes)):
        class_id = class_ids[i].item()
        if names[class_id] == "bottle":
            x1, y1, x2, y2 = boxes[i].tolist()
            label = f"{names[class_id]} {confidences[i].item():.2f}"

            # Draw rectangle (bounding box)
            color = (0, 255, 0)  # Green color for the box
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)

            # Put the label (class name and confidence) on the frame
            cv2.putText(frame, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Display the frame with the detected bounding boxes and threshold line
    cv2.putText(frame, f"Bottles Counted: {bottle_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('YOLO Object Detection - Bottle Counting with Threshold', frame)

    # Wait for key press (press 'q' to exit the video)
    k = cv2.waitKey(25)  # 10 ms delay for video processing
    if k == ord('q'):
        break

# Release video capture and close all windows
cap.release()
cv2.destroyAllWindows()
